# LM-KBC 2026 — self-contained run (val, Qwen/Qwen2.5-7B-Instruct)
Embeds the solution code; no external repo needed.


In [ ]:
import os
os.makedirs('solution', exist_ok=True)
print('GPU check:'); import subprocess; print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)


In [ ]:
!pip -q install 'transformers>=4.51.0' accelerate bitsandbytes 'numpy<2' 2>/dev/null
print('deps ok')


In [ ]:
%%writefile solution/relations.py
"""Per-relation configuration for the LM-KBC 2026 shared task.

Everything that differs between the six relations lives here: the precise scope
definition (copied from the official relation definitions — the model is told
these), how the answer should be shaped, how many objects to expect, generation
budget, and whether an empty answer is plausible (abstention lever).

This module has NO heavy dependencies so it can be imported and unit-tested on
a laptop with no GPU.
"""
from __future__ import annotations
from dataclasses import dataclass, field
from typing import List


@dataclass
class RelationSpec:
    name: str
    kind: str                      # "numeric" | "string"
    # One-sentence task shown to the model, phrased as an instruction.
    instruction: str
    # The official scope definition the ground truth was built against.
    definition: str
    # Natural-language description of the expected answer cardinality / shape.
    answer_shape: str
    allow_empty: bool              # can the correct answer legitimately be empty?
    multi_valued: bool             # can there be many objects?
    max_new_tokens: int            # generation budget (awardWonBy needs a lot)
    # how many few-shot exemplars to include from train (mix of empty/non-empty)
    few_shot: int = 6
    # unit hint for numeric relations
    unit: str = ""


RELATIONS: dict[str, RelationSpec] = {
    "countryLandBordersCountry": RelationSpec(
        name="countryLandBordersCountry",
        kind="string",
        instruction="List every country that shares a LAND border with the subject country.",
        definition=(
            "Countries (or comparable territories) that share a LAND border with the "
            "subject. Maritime/sea borders are EXCLUDED (e.g. Russia-Japan, Samoa-USA "
            "do NOT count). An island country with no land border has an EMPTY answer. "
            "Only currently-recognised states count."
        ),
        answer_shape="A list of country names (English common name). Empty list if the country is an island with no land neighbour.",
        allow_empty=True,
        multi_valued=True,
        max_new_tokens=128,
        few_shot=8,
    ),
    "personHasCityOfDeath": RelationSpec(
        name="personHasCityOfDeath",
        kind="string",
        instruction="Give the CITY where the subject person died.",
        definition=(
            "The CITY where the person died (city granularity, not country or region). "
            "If the person is still alive, or the city of death is unknown, the answer "
            "is EMPTY."
        ),
        answer_shape="Exactly one city name, or an empty list if the person is alive / city unknown.",
        allow_empty=True,
        multi_valued=False,
        max_new_tokens=48,
        few_shot=8,
    ),
    "companyTradesAtStockExchange": RelationSpec(
        name="companyTradesAtStockExchange",
        kind="string",
        instruction="List the stock exchange(s) on which the subject company's shares are publicly traded.",
        definition=(
            "The stock exchange(s) on which the company's shares are publicly traded. "
            "Multiple listings are possible. A private company or a subsidiary that is "
            "NOT separately listed has an EMPTY answer."
        ),
        answer_shape="A list of stock exchange names (e.g. 'New York Stock Exchange', 'Tokyo Stock Exchange'). Empty if not publicly listed.",
        allow_empty=True,
        multi_valued=True,
        max_new_tokens=64,
        few_shot=8,
    ),
    "awardWonBy": RelationSpec(
        name="awardWonBy",
        kind="string",
        instruction="List as many recipients/winners of the subject award as you know.",
        definition=(
            "Entities (people or organisations) that have received the SPECIFIC award "
            "named by the subject. Predecessor/successor awards with different names are "
            "DISTINCT and must not be bundled. Some awards have hundreds of recipients."
        ),
        answer_shape="A long list of recipient names. List every winner you can recall.",
        allow_empty=False,
        multi_valued=True,
        max_new_tokens=2048,
        few_shot=2,
    ),
    "hasCapacity": RelationSpec(
        name="hasCapacity",
        kind="numeric",
        instruction="Give the maximum spectator capacity of the subject venue, as a single integer number of people.",
        definition=(
            "The MAXIMUM spectator capacity of the venue, as an integer number of people "
            "(Wikidata P1083). When several capacities exist (seated vs total, "
            "before/after renovation), use the HIGHEST published capacity."
        ),
        answer_shape="A single integer (number of people), no commas, no units.",
        allow_empty=False,
        multi_valued=False,
        max_new_tokens=32,
        unit="people",
    ),
    "hasArea": RelationSpec(
        name="hasArea",
        kind="numeric",
        instruction="Give the total surface area of the subject in square kilometres (km²).",
        definition=(
            "The total surface area of the geographic entity in SQUARE KILOMETRES (km²). "
            "For countries use TOTAL area (land + inland water), Wikidata P2046. Convert "
            "from hectares/sq-miles to km² if needed."
        ),
        answer_shape="A single number in km² (may be a decimal), no units, no commas.",
        allow_empty=False,
        multi_valued=False,
        max_new_tokens=32,
        unit="km^2",
    ),
}

ALL_RELATIONS: List[str] = list(RELATIONS.keys())


In [ ]:
%%writefile solution/prompts.py
"""Prompt construction for the LM-KBC 2026 task.

Strategy:
  * One chat prompt per (subject, relation).
  * System message states the precise relation definition + scope so the model
    targets the same notion the ground truth was built against.
  * We force a small JSON output `{"answers": [...]}` so parsing is robust and
    the empty case is unambiguous (`{"answers": []}`).
  * Few-shot exemplars are drawn from the TRAIN split. For null-possible
    relations we deliberately mix empty and non-empty exemplars so the model
    learns that abstaining is a valid, expected answer (abstention is worth
    real points — predicting [] scores macro-F1 0.20 on val by itself).

No GPU / model dependencies here: a prompt is just a list of {role, content}
dicts, ready for `tokenizer.apply_chat_template`.
"""
from __future__ import annotations
import json
from typing import Dict, List

from relations import RelationSpec, RELATIONS


def _first_alias(obj_aliases: List[str]) -> str:
    return obj_aliases[0] if obj_aliases else ""


def gold_answer_list(row: Dict) -> List[str]:
    """Canonical (first-alias) answer strings for a train/val row."""
    return [_first_alias(o) for o in row["ObjectEntities"] if o]


def select_few_shot(spec: RelationSpec, train_rows: List[Dict], k: int) -> List[Dict]:
    """Pick up to k exemplars for this relation. For null-possible relations,
    interleave empty and non-empty examples so both behaviours are demonstrated.
    Deterministic (no RNG) for reproducibility."""
    rel_rows = [r for r in train_rows if r["Relation"] == spec.name]
    if not spec.allow_empty:
        return rel_rows[:k]
    empties = [r for r in rel_rows if len(r["ObjectEntities"]) == 0]
    nonempties = [r for r in rel_rows if len(r["ObjectEntities"]) > 0]
    out, i, j = [], 0, 0
    # alternate non-empty / empty, starting with non-empty
    while len(out) < k and (i < len(nonempties) or j < len(empties)):
        if i < len(nonempties):
            out.append(nonempties[i]); i += 1
        if len(out) < k and j < len(empties):
            out.append(empties[j]); j += 1
    return out


def _exemplar_answer_json(spec: RelationSpec, row: Dict) -> str:
    answers = gold_answer_list(row)
    if spec.name == "awardWonBy":
        # keep exemplar short: show a handful of winners, not hundreds
        answers = answers[:12]
    return json.dumps({"answers": answers}, ensure_ascii=False)


def system_message(spec: RelationSpec) -> str:
    lines = [
        "You are a precise knowledge-base construction engine. You answer purely "
        "from your own parametric knowledge (closed book).",
        "",
        f"TASK: {spec.instruction}",
        f"DEFINITION (the exact scope you are graded on): {spec.definition}",
        f"ANSWER SHAPE: {spec.answer_shape}",
        "",
        "Rules:",
        '- Reply with ONLY a single JSON object: {"answers": [...]}. No prose, no markdown.',
    ]
    if spec.kind == "numeric":
        lines.append(
            '- The list holds exactly one element: the number as a string, e.g. {"answers": ["35000"]}. '
            "No commas, no units, no ranges."
        )
    else:
        lines.append('- Each list element is one object string, e.g. {"answers": ["Haiti"]}.')
    if spec.allow_empty:
        lines.append(
            '- If the correct answer is genuinely empty (e.g. ' +
            ("an island country with no land border" if spec.name == "countryLandBordersCountry"
             else "the person is still alive or the city is unknown" if spec.name == "personHasCityOfDeath"
             else "the company is private / not separately listed") +
            '), reply {"answers": []}. Do NOT guess when unsure — a wrong guess is '
            "penalised, an empty answer is not."
        )
    else:
        lines.append("- Always give your single best answer; do not return an empty list.")
    return "\n".join(lines)


def user_message(spec: RelationSpec, subject: str) -> str:
    return f"Subject: {subject}\nRelation: {spec.name}\nAnswer:"


def build_messages(spec: RelationSpec, subject: str, train_rows: List[Dict]) -> List[Dict[str, str]]:
    msgs: List[Dict[str, str]] = [{"role": "system", "content": system_message(spec)}]
    for ex in select_few_shot(spec, train_rows, spec.few_shot):
        msgs.append({"role": "user", "content": user_message(spec, ex["SubjectEntity"])})
        msgs.append({"role": "assistant", "content": _exemplar_answer_json(spec, ex)})
    msgs.append({"role": "user", "content": user_message(spec, subject)})
    return msgs


def build_all(spec_name: str, subject: str, train_rows: List[Dict]) -> List[Dict[str, str]]:
    return build_messages(RELATIONS[spec_name], subject, train_rows)


In [ ]:
%%writefile solution/parsing.py
"""Robustly turn raw model text into a clean List[str] of object strings.

The model is asked for `{"answers": [...]}`, but real models drift: markdown
fences, trailing prose, bare lists, "None", numbers with commas/units, etc.
This module recovers a clean answer list and normalises numeric relations.

Pure-python, no model deps — fully unit-testable on a laptop.
"""
from __future__ import annotations
import json
import re
import string
import unicodedata
from typing import List, Optional

from relations import RelationSpec, RELATIONS

# Tokens that signal an explicit empty / abstain answer.
_EMPTY_TOKENS = {
    "", "none", "n/a", "na", "null", "nil", "unknown", "no answer", "empty",
    "not applicable", "no", "no land border", "still alive", "alive",
    "not publicly traded", "not listed", "private",
}


# --- normalization identical to evaluate.py (keep in sync) -------------------
def normalize_string(s: str) -> str:
    s = s.strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    for p in string.punctuation:
        s = s.replace(p, " ")
    return " ".join(s.split())


# --- JSON / list recovery ----------------------------------------------------
def _extract_answers_field(text: str) -> Optional[List]:
    """Find a JSON object containing an "answers" list, scanning all
    balanced-brace candidates and taking the first that parses."""
    # Strip code fences.
    text = re.sub(r"```(?:json)?", "", text)
    # Find every {...} candidate (greedy-ish, brace-balanced scan).
    for m in re.finditer(r"\{", text):
        start = m.start()
        depth = 0
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    blob = text[start:i + 1]
                    try:
                        obj = json.loads(blob)
                    except json.JSONDecodeError:
                        break
                    if isinstance(obj, dict) and "answers" in obj and isinstance(obj["answers"], list):
                        return obj["answers"]
                    break
    return None


def _extract_bare_list(text: str) -> Optional[List]:
    text2 = re.sub(r"```(?:json)?", "", text)
    m = re.search(r"\[.*\]", text2, re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            if isinstance(obj, list):
                return obj
        except json.JSONDecodeError:
            pass
    return None


def _fallback_lines(text: str) -> List[str]:
    """Last-resort: split prose into candidate items (comma / newline / bullets)."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    text = re.sub(r"^(the answer is|answer:|answers:)\s*", "", text, flags=re.I)
    parts = re.split(r"[\n,;]+|\s+and\s+", text)
    return [p.strip(" .-*•\t") for p in parts if p.strip(" .-*•\t")]


def raw_to_items(text: str) -> List[str]:
    """Recover a list of raw string items from any model output."""
    if text is None:
        return []
    items = _extract_answers_field(text)
    if items is None:
        items = _extract_bare_list(text)
    if items is None:
        items = _fallback_lines(text)
    out: List[str] = []
    for it in items:
        if isinstance(it, (int, float)):
            out.append(str(it))
        elif isinstance(it, str):
            out.append(it.strip())
        # ignore nested/other types
    return out


# --- numeric handling --------------------------------------------------------
_NUM_RE = re.compile(r"[-+]?\d[\d,]*\.?\d*")


def parse_number(s: str) -> Optional[str]:
    """Extract the first number from a string, handling commas, scaling words,
    and unit suffixes. Returns a canonical numeric string or None."""
    if not isinstance(s, str):
        s = str(s)
    low = s.lower().replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", low)
    if not m:
        return None
    val = float(m.group(0))
    tail = low[m.end():]
    # scaling words right after the number
    if re.match(r"\s*(thousand|k)\b", tail):
        val *= 1e3
    elif re.match(r"\s*(million|mn|m)\b", tail):
        val *= 1e6
    # hectare -> km^2 conversion only handled upstream via prompt; we trust km^2.
    # render: int if integral, else trimmed float
    if abs(val - round(val)) < 1e-9:
        return str(int(round(val)))
    return repr(val).rstrip("0").rstrip(".") if "." in repr(val) else str(val)


# --- main entrypoint ---------------------------------------------------------
def parse_prediction(relation: str, raw_text: str) -> List[str]:
    spec: RelationSpec = RELATIONS[relation]
    items = raw_to_items(raw_text)

    # drop explicit empty markers
    items = [it for it in items if normalize_string(it) not in _EMPTY_TOKENS]

    if spec.kind == "numeric":
        for it in items:
            n = parse_number(it)
            if n is not None:
                return [n]            # numeric relations want exactly one value
        return []

    # string relations: dedup by normalized form, preserve order & casing
    seen, out = set(), []
    for it in items:
        key = normalize_string(it)
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(it)

    if not spec.multi_valued:
        out = out[:1]                 # single-valued (cityOfDeath): keep best one
    return out


In [ ]:
%%writefile solution/decoding.py
"""Turn cached raw model samples into final ObjectEntities.

Kept separate from generation so the EXPENSIVE step (calling the model on
Kaggle) runs once and writes a raw-output cache, while the CHEAP step (parsing,
abstention, numeric aggregation) can be re-run locally for free as we iterate.

    raw cache row:  {"SubjectEntity", "Relation", "RawSamples": [str, ...]}
    prediction row: {"SubjectEntity", "Relation", "ObjectEntities": [str, ...]}
"""
from __future__ import annotations
from statistics import median
from typing import List

from relations import RELATIONS
from parsing import parse_prediction


def _fmt_num(x: float) -> str:
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return repr(x).rstrip("0").rstrip(".")


def decode_samples(relation: str, samples: List[str],
                   numeric_self_consistency: bool = True) -> List[str]:
    """Aggregate one (subject, relation)'s sampled completions into a final
    answer list. Numeric relations with >1 sample use the median (robust under
    the 5% tolerance metric); everything else decodes the first sample."""
    spec = RELATIONS[relation]
    if spec.kind == "numeric" and numeric_self_consistency and len(samples) > 1:
        nums = []
        for s in samples:
            p = parse_prediction(relation, s)
            if p:
                try:
                    nums.append(float(p[0]))
                except ValueError:
                    pass
        return [_fmt_num(median(nums))] if nums else []
    return parse_prediction(relation, samples[0] if samples else "")


def decode_cache(raw_rows: List[dict], numeric_self_consistency: bool = True) -> List[dict]:
    out = []
    for r in raw_rows:
        objs = decode_samples(r["Relation"], r.get("RawSamples", []), numeric_self_consistency)
        out.append({
            "SubjectEntity": r["SubjectEntity"],
            "Relation": r["Relation"],
            "ObjectEntities": objs,
        })
    return out


In [ ]:
%%writefile solution/generators.py
"""Generation backends behind a single interface.

    apply_template(messages) -> str         # one templated prompt
    generate(prompts, max_new_tokens, temperature, n) -> List[List[str]]
        outer list  = one entry per prompt
        inner list  = n sampled completions for that prompt (n>=1)

Backends:
  * VLLMGenerator   - fast batched inference (preferred on Kaggle 2xT4)
  * HFGenerator     - transformers + bitsandbytes 4-bit fallback
  * OracleMock      - no model; returns gold wrapped in noisy JSON (tests the
                      WHOLE prompt->parse->write->eval pipeline locally, no GPU)
  * EmptyMock       - returns {"answers": []} for everything (reproduces the
                      "predict nothing" 0.203 floor through the real pipeline)

Only the mock backends import nothing heavy, so the pipeline is exercisable on
a laptop. vLLM / torch are imported lazily inside the GPU backends.
"""
from __future__ import annotations
import json
from typing import Dict, List, Tuple

# delimiter used by mock backends to smuggle subject/relation through the
# (subject -> templated prompt -> generate) path. Plain ASCII, never appears
# in a real subject string.
MOCK_DELIM = " |##REL##| "


class BaseGenerator:
    def apply_template(self, messages: List[Dict[str, str]]) -> str:
        raise NotImplementedError

    def generate(self, prompts: List[str], max_new_tokens: int,
                 temperature: float = 0.0, n: int = 1) -> List[List[str]]:
        raise NotImplementedError


# --------------------------------------------------------------------------- #
# Mock backends (no GPU) — for local plumbing tests.
# --------------------------------------------------------------------------- #
def _field(messages: List[Dict[str, str]], prefix: str) -> str:
    for line in messages[-1]["content"].splitlines():
        if line.startswith(prefix):
            return line[len(prefix):].strip()
    return ""


class EmptyMock(BaseGenerator):
    def apply_template(self, messages):
        return "x"

    def generate(self, prompts, max_new_tokens, temperature=0.0, n=1):
        return [['{"answers": []}'] * n for _ in prompts]


class OracleMock(BaseGenerator):
    """Returns the gold answer (first alias) for each (subject,relation),
    wrapped in deliberately messy text (code fence + trailing prose) so the
    parser is exercised. gold_map: {(subject, relation): [first_alias,...]}."""
    def __init__(self, gold_map: Dict[Tuple[str, str], List[str]]):
        self.gold_map = gold_map

    def apply_template(self, messages):
        return _field(messages, "Subject:") + MOCK_DELIM + _field(messages, "Relation:")

    def generate(self, prompts, max_new_tokens, temperature=0.0, n=1):
        out = []
        for key in prompts:
            subj, _, rel = key.partition(MOCK_DELIM)
            gold = self.gold_map.get((subj, rel), [])
            blob = "```json\n" + json.dumps({"answers": gold}, ensure_ascii=False) + "\n```\nDone."
            out.append([blob] * n)
        return out


# --------------------------------------------------------------------------- #
# vLLM backend (preferred on Kaggle).
# --------------------------------------------------------------------------- #
class VLLMGenerator(BaseGenerator):
    def __init__(self, model_path: str, tensor_parallel_size: int = 1,
                 quantization: str | None = None, max_model_len: int = 4096,
                 gpu_memory_utilization: float = 0.92, dtype: str = "auto"):
        from vllm import LLM                       # lazy import
        from transformers import AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        self._tmpl_kwargs = {"tokenize": False, "add_generation_prompt": True}
        # Qwen3 etc. expose a thinking switch; disable it for terse JSON output.
        try:
            if "enable_thinking" in self.tokenizer.apply_chat_template.__doc__ or True:
                self._tmpl_kwargs["enable_thinking"] = False
                _ = self.tokenizer.apply_chat_template(
                    [{"role": "user", "content": "hi"}], **self._tmpl_kwargs)
        except TypeError:
            self._tmpl_kwargs.pop("enable_thinking", None)
        self.llm = LLM(
            model=model_path, tensor_parallel_size=tensor_parallel_size,
            quantization=quantization, max_model_len=max_model_len,
            gpu_memory_utilization=gpu_memory_utilization, dtype=dtype,
            trust_remote_code=True,
        )

    def apply_template(self, messages):
        return self.tokenizer.apply_chat_template(messages, **self._tmpl_kwargs)

    def generate(self, prompts, max_new_tokens, temperature=0.0, n=1):
        from vllm import SamplingParams
        sp = SamplingParams(n=n, temperature=temperature,
                            top_p=0.95 if temperature > 0 else 1.0,
                            max_tokens=max_new_tokens)
        outs = self.llm.generate(prompts, sp)
        return [[o.text for o in req.outputs] for req in outs]


# --------------------------------------------------------------------------- #
# transformers + bitsandbytes 4-bit fallback.
# --------------------------------------------------------------------------- #
class HFGenerator(BaseGenerator):
    def __init__(self, model_path: str, load_in_4bit: bool = True,
                 device_map: str = "auto", dtype: str = "float16"):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        self.torch = torch
        td = getattr(torch, dtype)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        qcfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        ) if load_in_4bit else None
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path, device_map=device_map, quantization_config=qcfg,
            torch_dtype=td, trust_remote_code=True)
        self.model.eval()

    def apply_template(self, messages):
        try:  # Qwen3 etc.: force thinking OFF so we get terse JSON, not <think> blocks
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)

    def generate(self, prompts, max_new_tokens, temperature=0.0, n=1, batch_size=8):
        torch = self.torch
        results: List[List[str]] = []
        do_sample = temperature > 0
        eff_bs = max(1, batch_size // max(1, n))   # n return-seqs per prompt -> shrink batch to fit VRAM
        for i in range(0, len(prompts), eff_bs):
            batch = prompts[i:i + eff_bs]
            enc = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(self.model.device)
            with torch.no_grad():
                gen = self.model.generate(
                    **enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                    temperature=temperature if do_sample else None,
                    top_p=0.95 if do_sample else None,
                    num_return_sequences=n,
                    pad_token_id=self.tokenizer.pad_token_id)
            gen = gen[:, enc["input_ids"].shape[1]:]
            texts = self.tokenizer.batch_decode(gen, skip_special_tokens=True)
            for j in range(len(batch)):
                results.append(texts[j * n:(j + 1) * n])
        return results


In [ ]:
%%writefile solution/run.py
"""End-to-end runner: (subject, relation) -> predictions.jsonl

    # local plumbing (no GPU): oracle should ~match gold, empty -> the 0.203 floor
    python solution/run.py --backend oracle --input dataset2026/data/val.jsonl -o /tmp/oracle.jsonl
    python solution/run.py --backend empty  --input dataset2026/data/val.jsonl -o /tmp/empty.jsonl

    # Kaggle (2xT4) with vLLM:
    python solution/run.py --backend vllm --model Qwen/Qwen2.5-14B-Instruct \
        --tp 2 --input dataset2026/data/val.jsonl -o preds_val.jsonl

Numeric relations use self-consistency (sample N, take the median) — robust to
outliers and well-suited to the 5% tolerance metric. String relations default
to greedy decoding.
"""
from __future__ import annotations
import argparse
import json
import os
import sys
from collections import defaultdict
from statistics import median
from typing import Dict, List

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from relations import RELATIONS, ALL_RELATIONS          # noqa: E402
from prompts import build_messages, gold_answer_list     # noqa: E402
from decoding import decode_samples                       # noqa: E402


def read_jsonl(path: str) -> List[Dict]:
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


def build_generator(args, gold_map):
    if args.backend == "oracle":
        from generators import OracleMock
        return OracleMock(gold_map)
    if args.backend == "empty":
        from generators import EmptyMock
        return EmptyMock()
    if args.backend == "vllm":
        from generators import VLLMGenerator
        return VLLMGenerator(args.model, tensor_parallel_size=args.tp,
                             quantization=args.quantization,
                             max_model_len=args.max_model_len)
    if args.backend == "hf":
        from generators import HFGenerator
        return HFGenerator(args.model, load_in_4bit=not args.no_4bit)
    raise ValueError(args.backend)


def run(args):
    rows = read_jsonl(args.input)
    train = read_jsonl(args.train)
    if args.relations:
        keep = set(args.relations.split(","))
        rows = [r for r in rows if r["Relation"] in keep]
    if args.limit:
        # keep a balanced-ish slice: first N per relation
        per = defaultdict(int); sliced = []
        for r in rows:
            if per[r["Relation"]] < args.limit:
                sliced.append(r); per[r["Relation"]] += 1
        rows = sliced

    gold_map = {(r["SubjectEntity"], r["Relation"]): gold_answer_list(r)
                for r in rows} if args.backend == "oracle" else {}

    gen = build_generator(args, gold_map)

    by_rel: Dict[str, List[Dict]] = defaultdict(list)
    for r in rows:
        by_rel[r["Relation"]].append(r)

    raw_rows: List[Dict] = []          # the expensive-to-produce generation cache
    predictions: List[Dict] = []
    for rel, rel_rows in by_rel.items():
        spec = RELATIONS[rel]
        prompts = [gen.apply_template(build_messages(spec, r["SubjectEntity"], train))
                   for r in rel_rows]

        numeric_sc = spec.kind == "numeric" and args.sc_samples > 1 and args.backend in ("vllm", "hf")
        if numeric_sc:
            comps = gen.generate(prompts, spec.max_new_tokens,
                                 temperature=args.sc_temperature, n=args.sc_samples)
        else:
            comps = gen.generate(prompts, spec.max_new_tokens, temperature=0.0, n=1)

        for r, sample_list in zip(rel_rows, comps):
            raw_rows.append({
                "SubjectEntity": r["SubjectEntity"],
                "Relation": rel,
                "RawSamples": sample_list,
            })
            predictions.append({
                "SubjectEntity": r["SubjectEntity"],
                "Relation": rel,
                "ObjectEntities": decode_samples(rel, sample_list, numeric_self_consistency=numeric_sc),
            })
        print(f"  [{rel}] {len(rel_rows)} rows done", file=sys.stderr)

    # raw cache: re-decode locally with decode.py instead of re-running the model
    raw_path = args.raw_out or (os.path.splitext(args.output)[0] + ".raw.jsonl")
    with open(raw_path, "w") as f:
        for r in raw_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(args.output, "w") as f:
        for p in predictions:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"wrote {len(predictions)} predictions -> {args.output}", file=sys.stderr)
    print(f"wrote raw generation cache -> {raw_path}", file=sys.stderr)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--backend", required=True, choices=["vllm", "hf", "oracle", "empty"])
    ap.add_argument("--model", default=None)
    ap.add_argument("-i", "--input", required=True)
    ap.add_argument("--train", default=None)
    ap.add_argument("-o", "--output", required=True)
    ap.add_argument("--raw-out", default=None, dest="raw_out",
                    help="path for the raw generation cache (default: <output>.raw.jsonl)")
    ap.add_argument("--tp", type=int, default=1, help="vLLM tensor_parallel_size")
    ap.add_argument("--quantization", default=None, help="e.g. awq, gptq, bitsandbytes")
    ap.add_argument("--max-model-len", type=int, default=4096, dest="max_model_len")
    ap.add_argument("--no-4bit", action="store_true", help="(hf) disable 4-bit")
    ap.add_argument("--sc-samples", type=int, default=5, help="self-consistency samples for numeric relations")
    ap.add_argument("--sc-temperature", type=float, default=0.7)
    ap.add_argument("--relations", default=None, help="comma-separated subset")
    ap.add_argument("--limit", type=int, default=0, help="max rows per relation (debug)")
    args = ap.parse_args()
    if args.train is None:
        args.train = os.path.join(os.path.dirname(args.input), "train.jsonl")
    run(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile solution/eval.py
"""Score a predictions file against gold, with a per-relation breakdown and a
comparison against the 'predict nothing' floor.

    python solution/eval.py -p preds_val.jsonl -g dataset2026/data/val.jsonl
"""
from __future__ import annotations
import argparse
import importlib.util
import json
import os

REPO_EVAL = os.path.join(os.path.dirname(__file__), "..", "dataset2026", "evaluate.py")


def load_evaluator():
    spec = importlib.util.spec_from_file_location("ev", REPO_EVAL)
    ev = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ev)
    return ev


def macro_table(ev, pred_rows, gt_rows):
    sc = ev.evaluate_per_sr_pair(pred_rows, gt_rows, ev.RELATION_TYPE, 0.05)
    return ev.macro_average_per_relation(sc)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("-p", "--predictions", required=True)
    ap.add_argument("-g", "--ground_truth", required=True)
    args = ap.parse_args()

    ev = load_evaluator()
    gt = ev.read_jsonl_file(args.ground_truth)
    pred = ev.read_jsonl_file(args.predictions)

    # empty-prediction floor on the same gold
    empty = [{"SubjectEntity": r["SubjectEntity"], "Relation": r["Relation"],
              "ObjectEntities": []} for r in gt]

    me = macro_table(ev, empty, gt)
    mp = macro_table(ev, pred, gt)

    per_rels = [k for k in mp if k != "*** All Relations ***"]
    print(f"{'relation':32} {'macro-f1':>9} {'(empty)':>9} {'Δ':>7}   {'P':>5} {'R':>5}")
    print("-" * 74)
    for rel in per_rels:
        f1 = mp[rel]["macro-f1"]; ef1 = me[rel]["macro-f1"]
        p = mp[rel]["macro-p"]; r = mp[rel]["macro-r"]
        flag = "  <-- below empty!" if f1 + 1e-9 < ef1 else ""
        print(f"{rel:32} {f1:>9.3f} {ef1:>9.3f} {f1-ef1:>+7.3f}   {p:>5.2f} {r:>5.2f}{flag}")
    print("-" * 74)
    # Two summaries. The 2025 edition ranked by AVG-OF-RELATIONS (each relation
    # 1/6 weight) — treat that as the headline. All-pairs is what evaluate.py prints.
    avg6 = sum(mp[r]["macro-f1"] for r in per_rels) / len(per_rels)
    eavg6 = sum(me[r]["macro-f1"] for r in per_rels) / len(per_rels)
    ap = mp["*** All Relations ***"]["macro-f1"]
    eap = me["*** All Relations ***"]["macro-f1"]
    print(f"{'AVG of relations (likely rank)':32} {avg6:>9.3f} {eavg6:>9.3f} {avg6-eavg6:>+7.3f}")
    print(f"{'All-pairs (evaluate.py)':32} {ap:>9.3f} {eap:>9.3f} {ap-eap:>+7.3f}")


if __name__ == "__main__":
    main()


In [ ]:
!git clone -q --depth 1 https://github.com/lm-kbc/dataset2026.git
print('data:', os.listdir('dataset2026/data'))


In [ ]:
MODEL   = 'Qwen/Qwen2.5-7B-Instruct'
BACKEND = 'hf'
SPLIT   = 'val'
SC      = 5


In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL, trust_remote_code=True)
print('config loaded for', MODEL)


In [ ]:
INPUT='dataset2026/data/%s.jsonl'%SPLIT; TRAIN='dataset2026/data/train.jsonl'; OUT='/kaggle/working/preds_%s.jsonl'%SPLIT
cmd=f'cd solution && python run.py --backend {BACKEND} --model {MODEL} -i ../{INPUT} --train ../{TRAIN} -o {OUT} --sc-samples {SC}'
print(cmd)
import subprocess,sys
p=subprocess.run(cmd,shell=True,capture_output=True,text=True)
print(p.stdout[-3000:]); print('STDERR:', p.stderr[-3000:])


In [ ]:
if SPLIT=='val':
    r=subprocess.run(f'cd solution && python eval.py -p {OUT} -g ../{INPUT}',shell=True,capture_output=True,text=True)
    print(r.stdout); print(r.stderr[-1500:])


In [ ]:
# artifacts: /kaggle/working/preds_{split}.jsonl + .raw.jsonl are auto-saved as kernel output
print(sorted(f for f in os.listdir('/kaggle/working') if f.endswith('.jsonl')))
